# Early metro data exploration

This notebook is an exploratory predecessor to [the forecasting study](https://github.com/jeck5iv/highload-context-ensemble-forecasting).

It expects private operational files and cannot be rerun from this repository alone. Saved outputs have been cleared to avoid redistributing source records. The original exploratory code is retained; use the study repository for the final model, evaluation and tests.


In [ ]:
from pathlib import Path
import pandas as pd

DIR = Path("./data")

In [ ]:
GDS_APPLICATION_202503251731 = pd.read_csv(DIR / "GDS_APPLICATION_202503251731.csv", sep=";", nrows=50000)
display(GDS_APPLICATION_202503251731.head(10))

In [ ]:
PASS_ALL_202503242210 = pd.read_csv(DIR / "PASS_ALL_202503242210.csv", sep=";", nrows=50000)
display(PASS_ALL_202503242210.head(10))

In [ ]:
REF_TRANSPORT_WAY_202503251803 = pd.read_csv(DIR / "REF_TRANSPORT_WAY_202503251803.csv", sep=";", nrows=50000)
display(REF_TRANSPORT_WAY_202503251803.head(10))

In [ ]:
GDS_GOODS_202503251844 = pd.read_csv(DIR / "GDS_GOODS_202503251844.csv", sep=";", nrows=50000)
display(GDS_GOODS_202503251844.head(10))

In [ ]:
REF_PSG_PLACES_202503251822 = pd.read_csv(DIR / "REF_PSG_PLACES_202503251822.csv", sep=";", nrows=50000)
display(REF_PSG_PLACES_202503251822.head(10))

In [ ]:
TRN_TYPE_202503251753 = pd.read_csv(DIR / "TRN_TYPE_202503251753.csv", sep=";", nrows=50000)
display(TRN_TYPE_202503251753.head(10))

In [ ]:
GDS_TRANSFER_202503251738 = pd.read_csv(DIR / "GDS_TRANSFER_202503251738.csv", sep=";", nrows=50000)
display(GDS_TRANSFER_202503251738.head(10))

In [ ]:
REF_TRANSPORT_TYPE_202503251727 = pd.read_csv(DIR / "REF_TRANSPORT_TYPE_202503251727.csv", sep=";", nrows=50000)
display(REF_TRANSPORT_TYPE_202503251727.head(10))

In [ ]:
%pip install duckdb pyarrow


In [ ]:
# PASS_ALL_202503242210 = pd.read_csv(DIR / "PASS_ALL_202503242210.csv", sep=";")
# display(PASS_ALL_202503242210.head(10))

In [ ]:
from pathlib import Path
import duckdb

KEEP_COLS = ["TRAN_DATE", "TRANSPORT_TYPE_ID", "PLACE_ID", "VALIDATION_MODE", "APP_ID"]

KEEP_FILTERS = [
    "TRANSPORT_TYPE_ID = 1",
    "PLACE_ID IN (1623, 1624)",
]

def _needs_rebuild(target: Path, *sources: Path) -> bool:
    if (not target.exists()) or target.stat().st_size == 0:
        return True
    t = target.stat().st_mtime
    return any(s.exists() and s.stat().st_mtime > t for s in sources)

def build_full_parquet(csv_path: Path, full_parquet: Path, sep: str = ";") -> None:
    con = duckdb.connect()
    con.execute("PRAGMA enable_progress_bar=true;")
    con.execute("PRAGMA progress_bar_time=1;")

    print(f"[DuckDB] Конвертирую CSV -> FULL Parquet:\n  {csv_path}\n  -> {full_parquet}")
    con.execute(f"""
        COPY (
          SELECT * FROM read_csv_auto('{str(csv_path)}', delim='{sep}')
        ) TO '{str(full_parquet)}' (FORMAT PARQUET);
    """)
    con.close()

def build_slim_parquet(source_parquet: Path, slim_parquet: Path) -> None:
    cols_sql = ", ".join(KEEP_COLS)

    con = duckdb.connect()
    con.execute("PRAGMA enable_progress_bar=true;")
    con.execute("PRAGMA progress_bar_time=1;")

    print(f"[DuckDB] Делаю SLIM Parquet (5 колонок, без фильтров):\n  {source_parquet}\n  -> {slim_parquet}")
    con.execute(f"""
        COPY (
          SELECT {cols_sql}
          FROM read_parquet('{str(source_parquet)}')
        ) TO '{str(slim_parquet)}' (FORMAT PARQUET);
    """)
    con.close()

def build_slim_filtered_parquet(source_parquet: Path, slim_filt_parquet: Path, filters: list[str]) -> None:
    cols_sql = ", ".join(KEEP_COLS)
    where_sql = " AND ".join(f"({f})" for f in filters) if filters else "TRUE"

    con = duckdb.connect()
    con.execute("PRAGMA enable_progress_bar=true;")
    con.execute("PRAGMA progress_bar_time=1;")

    print(f"[DuckDB] Делаю SLIM+FILTERED Parquet:\n  {source_parquet}\n  -> {slim_filt_parquet}\n  WHERE {where_sql}")
    con.execute(f"""
        COPY (
          SELECT {cols_sql}
          FROM read_parquet('{str(source_parquet)}')
          WHERE {where_sql}
        ) TO '{str(slim_filt_parquet)}' (FORMAT PARQUET);
    """)
    con.close()

def load_parquet_to_pandas(parquet_path: Path):
    con = duckdb.connect()
    con.execute("PRAGMA enable_progress_bar=true;")
    con.execute("PRAGMA progress_bar_time=1;")

    print("[DuckDB] Выгружаю Parquet в pandas DataFrame (с прогрессом)...")
    df = con.execute(f"SELECT * FROM read_parquet('{str(parquet_path)}')").df()
    con.close()
    return df

def prepare_and_load(
    csv_path: Path,
    out_dir: Path | None = None,
    sep: str = ";",
    make_slim_unfiltered: bool = False,
    filters: list[str] | None = None,
):
    csv_path = Path(csv_path)
    out_dir = Path(out_dir) if out_dir is not None else csv_path.parent

    stem = csv_path.stem
    full_parquet = out_dir / f"{stem}_full.parquet"
    slim_parquet = out_dir / f"{stem}_slim.parquet"
    slim_filt_parquet = out_dir / f"{stem}_slim_filt.parquet"

    if filters is None:
        filters = KEEP_FILTERS

    if _needs_rebuild(full_parquet, csv_path):
        build_full_parquet(csv_path, full_parquet, sep=sep)
    else:
        print(f"[DuckDB] FULL Parquet уже готов:\n  {full_parquet}")

    if make_slim_unfiltered:
        if _needs_rebuild(slim_parquet, full_parquet):
            build_slim_parquet(full_parquet, slim_parquet)
        else:
            print(f"[DuckDB] SLIM Parquet уже готов:\n  {slim_parquet}")

    if _needs_rebuild(slim_filt_parquet, full_parquet):
        build_slim_filtered_parquet(full_parquet, slim_filt_parquet, filters)
    else:
        print(f"[DuckDB] SLIM+FILTERED Parquet уже готов:\n  {slim_filt_parquet}")

    df = load_parquet_to_pandas(slim_filt_parquet)
    return df, full_parquet, slim_filt_parquet


PASS_ALL_202503242210, FULL_PQ, SLIM_FILT_PQ = prepare_and_load(
    csv_path=DIR / "PASS_ALL_202503242210.csv",
    sep=";",
    make_slim_unfiltered=False,
)


In [ ]:
display(PASS_ALL_202503242210.head(10))

In [ ]:
n_rows = len(PASS_ALL_202503242210)
print("Rows:", n_rows)

n_unique = PASS_ALL_202503242210.nunique(dropna=False)
display(n_unique)


In [ ]:
PASS_ALL_202503242210 = PASS_ALL_202503242210.sort_values("TRAN_DATE")


In [ ]:
dates_list = PASS_ALL_202503242210["TRAN_DATE"]
dates_list

In [ ]:
# PLACE_ID = PASS_ALL_202503242210.loc[PASS_ALL_202503242210["PLACE_ID"] < 200, "PLACE_ID"]
# PLACE_ID


In [ ]:

PASS_ALL_202503242210


In [ ]:
df = PASS_ALL_202503242210[["TRAN_DATE"]].copy()


In [ ]:
import pandas as pd

df["TRAN_DATE"] = pd.to_datetime(df["TRAN_DATE"], errors="coerce")
df["hour_bucket"] = df["TRAN_DATE"].dt.floor("H")
hourly_counts = (
    df
    .groupby("hour_bucket")
    .size()
    .reset_index(name="count")
    .sort_values("hour_bucket")
)

display(hourly_counts.head(10))


In [ ]:
display(hourly_counts.head(10))

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(hourly_counts["hour_bucket"], hourly_counts["count"])
plt.xticks(rotation=45)
plt.xlabel("Time")
plt.ylabel("Number of validations")
plt.title("Hourly passenger flow")
plt.tight_layout()
plt.show()


In [ ]:
df = hourly_counts.copy()
df = df.sort_values("hour_bucket").reset_index(drop=True)
df["lag_1"] = df["count"].shift(1)
df["lag_2"] = df["count"].shift(2)
df = df.dropna().reset_index(drop=True)

In [ ]:
test_date = df["hour_bucket"].dt.date.max()

train = df[df["hour_bucket"].dt.date < test_date]
test  = df[df["hour_bucket"].dt.date == test_date]

print("Train size:", len(train))
print("Test size:", len(test))

In [ ]:
features = ["lag_1", "lag_2"]

X_train = train[features]
y_train = train["count"]

X_test = test[features]
y_test = test["count"]

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor(
    loss="absolute_error",
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(test["hour_bucket"], y_test, label="Real")
plt.plot(test["hour_bucket"], y_pred, label="Predicted")
plt.xticks(rotation=45)
plt.xlabel("Time")
plt.ylabel("Passengers")
plt.title("1-hour ahead forecast (Boosting)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
q80 = hourly_counts["count"].quantile(0.8)
print("0.8 quantile:", q80)
hourly_counts["highload"] = (
    hourly_counts["count"] > q80
).astype(int)

In [ ]:
display(hourly_counts.head())
print(hourly_counts["highload"].value_counts())


In [ ]:
import matplotlib.pyplot as plt

plt.figure()

plt.plot(hourly_counts["hour_bucket"],
         hourly_counts["highload"])

plt.xticks(rotation=45)
plt.xlabel("Time")
plt.ylabel("Highload (0/1)")
plt.title("Highload Hours (above 0.8 quantile)")

plt.tight_layout()
plt.show()
